In [1]:
# ==============================================================================
# CELL 0: FIX TORCHAUDIO CUDA MISMATCH & INSTALL DEPENDENCIES
# ==============================================================================
# 1. Remove mismatched torchaudio or force reinstall compatible torch/torchaudio
!pip uninstall -y torchaudio

# 2. Install/Upgrade vLLM and HTTP dependencies
!pip install -q --no-cache-dir "vllm>=0.6.0" httpx

Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 48.5 MB/s eta 0:00:00


In [2]:
# ==============================================================================
# CELL 1: ENVIRONMENT CLEANUP & VLLM SERVER LAUNCH
# ==============================================================================
import os
import sys
import time
import subprocess
import urllib.request
import gc
import torch

# 1. Terminate any previous background processes and clear VRAM/ports
subprocess.run("pkill -9 -f vllm", shell=True)
subprocess.run("fuser -k 8000/tcp", shell=True)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 2. Server Configuration
# Note: Ensure LOCKED_MODEL matches your exact target model
LOCKED_MODEL = "Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int8"
LOG_FILE = "server.log"
PORT = 8000

# Using sys.executable guarantees it executes with the same Python environment
# where `pip install vllm` was run.
SERVER_ARGS = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", LOCKED_MODEL,
    "--quantization", "gptq",
    "--dtype", "float16",
    "--max-model-len", "2048",
    "--gpu-memory-utilization", "0.75",
    "--enforce-eager",
    "--port", str(PORT),
    "--trust-remote-code"
]

print(f"Launching vLLM server for model: {LOCKED_MODEL}...")
logf = open(LOG_FILE, "wb")
server_proc = subprocess.Popen(
    SERVER_ARGS,
    stdout=logf,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

# 3. Health Polling Loop
url = f"http://localhost:{PORT}/v1/models"
start_time = time.time()
print("Polling server health endpoint (this may take 1-2 minutes while weights load)...", end="", flush=True)

healthy = False
while time.time() - start_time < 300:  # 5-minute timeout for downloading & loading weights
    if server_proc.poll() is not None:
        print("\n[ERROR] Server process terminated unexpectedly!")
        break
    try:
        with urllib.request.urlopen(url) as response:
            if response.status == 200:
                print("\nServer is HEALTHY!")
                healthy = True
                break
    except Exception:
        pass
    time.sleep(3)
    print(".", end="", flush=True)

if not healthy:
    print("\n--- SERVER LOG OUTPUT ---")
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, "r") as f:
            print(f.read())
    raise RuntimeError("Server failed to start. Review server.log above for details.")

Launching vLLM server for model: Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int8...
Polling server health endpoint (this may take 1-2 minutes while weights load).......................................
Server is HEALTHY!


In [3]:
# ==============================================================================
# CELL 2: PREPARE PROMPTS & EXACT REFERENCE BENCHMARK HARNESS
# ==============================================================================
import os

# 1. Write prompts.txt
PROMPTS_CONTENT = """What is a GPU?
Define tokens per second in one line.
Explain the difference between prefill and decode in two sentences.
List three reasons decode is memory-bound rather than compute-bound.
Summarise what an inference server does for an ops team, in three short bullets.
Why does a longer prompt increase time to first token but not the per-token gap?
Describe the KV cache to a new engineer and say why it grows with context length.
Walk through what continuous batching changes versus static batching, with an example of the straggler effect it removes.
Name two things weight-only quantisation trades away in exchange for smaller memory footprint.
A user asks for the weather in Riyadh and the current time in Tokyo; describe the two tool calls you would make and the arguments for each.
Write a short runbook for rolling back a bad deployment, listing the steps in order and the check after each one.
Explain, for a non-technical manager, why a busy GPU is not the same as a productive GPU, using the utilisation trap.
Compare fp16 and int4 for serving a 1.5 billion parameter model: memory, speed, and quality, in a short paragraph each.
Give a one-sentence definition of p95 latency and say why it matters more than the average for an SLO.
Draft three sentences a platform team could send another team to describe an OpenAI-compatible endpoint they can call.
Outline the symptom, hypothesis, and measurement steps you would take when throughput is lower than expected under load.
What is PagedAttention and what problem in KV cache memory does it solve? Answer in two sentences.
Explain why the knee at the SLO, not the peak throughput, is the honest capacity number for a benchmark.
Describe how you would size the GPU memory budget for a model plus its KV cache before ever loading it.
Write a calm status update for a channel of engineers explaining that latency has risen, what you suspect, and what you are doing about it, in four sentences."""

with open("prompts.txt", "w", encoding="utf-8") as f:
    f.write(PROMPTS_CONTENT.strip())

print("prompts.txt written successfully.")

# 2. Write exact reference bench.py
BENCH_SCRIPT = """\"\"\"Benchmark harness for the serving stack (week 3 day 5 reference).

Sweeps concurrency levels against an OpenAI-compatible endpoint and reports the
numbers the week-3 lab is graded on: tokens/sec, TTFT p50/p95, end-to-end
latency p95, and error counts per level.
\"\"\"

from __future__ import annotations

import argparse
import asyncio
import json
import os
import statistics
import time
from dataclasses import dataclass, field
from typing import Optional

import httpx

# --------------------------------------------------------------------------- #
# Per-request measurement                                                      #
# --------------------------------------------------------------------------- #

@dataclass
class RequestResult:
    ok: bool
    ttft_s: Optional[float] = None          # time to first content chunk
    latency_s: Optional[float] = None       # time to [DONE]
    completion_tokens: int = 0              # counted from streamed chunks
    error: Optional[str] = None


async def _one_request(
    client: httpx.AsyncClient,
    base_url: str,
    model: str,
    prompt: str,
    max_tokens: int,
) -> RequestResult:
    body = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "stream": True,
        "temperature": 0.0,
    }
    url = base_url.rstrip("/") + "/v1/chat/completions"
    start = time.perf_counter()
    ttft: Optional[float] = None
    tokens = 0

    try:
        async with client.stream("POST", url, json=body) as response:
            if response.status_code != 200:
                text = (await response.aread()).decode("utf-8", "replace")[:200]
                return RequestResult(
                    ok=False, error=f"HTTP {response.status_code}: {text}"
                )
            async for line in response.aiter_lines():
                if not line or not line.startswith("data: "):
                    continue
                data = line[len("data: "):]
                if data == "[DONE]":
                    break
                try:
                    chunk = json.loads(data)
                except json.JSONDecodeError:
                    continue
                delta = chunk.get("choices", [{}])[0].get("delta", {})
                content = delta.get("content")
                if content:
                    if ttft is None:
                        ttft = time.perf_counter() - start
                    tokens += 1
        latency = time.perf_counter() - start
        return RequestResult(
            ok=True, ttft_s=ttft, latency_s=latency, completion_tokens=tokens
        )
    except Exception as exc:
        return RequestResult(ok=False, error=f"{type(exc).__name__}: {exc}")


# --------------------------------------------------------------------------- #
# Per-level sweep                                                              #
# --------------------------------------------------------------------------- #

@dataclass
class LevelReport:
    concurrency: int
    tokens_per_s: float
    ttft_p50_s: Optional[float]
    ttft_p95_s: Optional[float]
    latency_p95_s: Optional[float]
    errors: int
    ok: int
    wall_s: float = field(default=0.0)


def _percentile(values: list[float], pct: float) -> Optional[float]:
    if not values:
        return None
    ordered = sorted(values)
    if len(ordered) == 1:
        return round(ordered[0], 4)
    rank = max(1, int(round(pct / 100.0 * len(ordered))))
    rank = min(rank, len(ordered))
    return round(ordered[rank - 1], 4)


async def _run_level(
    client: httpx.AsyncClient,
    base_url: str,
    model: str,
    prompts: list[str],
    concurrency: int,
    requests_per_level: int,
    max_tokens: int,
) -> LevelReport:
    await _one_request(client, base_url, model, prompts[0], max_tokens)

    semaphore = asyncio.Semaphore(concurrency)

    async def _guarded(index: int) -> RequestResult:
        async with semaphore:
            prompt = prompts[index % len(prompts)]
            return await _one_request(client, base_url, model, prompt, max_tokens)

    level_start = time.perf_counter()
    results = await asyncio.gather(
        *(_guarded(i) for i in range(requests_per_level))
    )
    wall = time.perf_counter() - level_start

    ok = [r for r in results if r.ok]
    errors = len(results) - len(ok)
    ttfts = [r.ttft_s for r in ok if r.ttft_s is not None]
    latencies = [r.latency_s for r in ok if r.latency_s is not None]
    total_tokens = sum(r.completion_tokens for r in ok)

    tokens_per_s = round(total_tokens / wall, 2) if wall > 0 else 0.0

    return LevelReport(
        concurrency=concurrency,
        tokens_per_s=tokens_per_s,
        ttft_p50_s=_percentile(ttfts, 50),
        ttft_p95_s=_percentile(ttfts, 95),
        latency_p95_s=_percentile(latencies, 95),
        errors=errors,
        ok=len(ok),
        wall_s=round(wall, 3),
    )


# --------------------------------------------------------------------------- #
# Orchestration                                                                #
# --------------------------------------------------------------------------- #

def _load_prompts(path: str) -> list[str]:
    with open(path, encoding="utf-8") as handle:
        prompts = [line.strip() for line in handle if line.strip()]
    if not prompts:
        raise SystemExit(f"prompt file {path!r} has no non-empty lines")
    return prompts


def _print_table(levels: list[LevelReport]) -> None:
    header = (
        f"{'conc':>4}  {'tok/s':>8}  {'ttft_p50':>9}  {'ttft_p95':>9}  "
        f"{'lat_p95':>8}  {'ok':>4}  {'err':>4}"
    )
    print(header)
    print("-" * len(header))
    for lv in levels:
        def fmt(value: Optional[float]) -> str:
            return f"{value:.3f}" if value is not None else "  n/a"
        print(
            f"{lv.concurrency:>4}  {lv.tokens_per_s:>8.2f}  "
            f"{fmt(lv.ttft_p50_s):>9}  {fmt(lv.ttft_p95_s):>9}  "
            f"{fmt(lv.latency_p95_s):>8}  {lv.ok:>4}  {lv.errors:>4}"
        )


def _write_report(out_path: str, run_record: dict) -> None:
    document: dict = {"runs": []}
    if os.path.exists(out_path):
        try:
            with open(out_path, encoding="utf-8") as handle:
                existing = json.load(handle)
            if isinstance(existing, dict) and isinstance(existing.get("runs"), list):
                document = existing
        except (json.JSONDecodeError, OSError):
            document = {"runs": []}
    document["runs"].append(run_record)
    with open(out_path, "w", encoding="utf-8") as handle:
        json.dump(document, handle, indent=2)


async def _sweep(args: argparse.Namespace) -> tuple[dict, list[LevelReport]]:
    prompts = _load_prompts(args.prompt_file)
    concurrency_levels = [int(c) for c in args.concurrency.split(",") if c.strip()]

    timeout = httpx.Timeout(args.timeout, connect=10.0)
    limits = httpx.Limits(max_connections=max(concurrency_levels) + 4)
    levels: list[LevelReport] = []

    headers = {}
    if getattr(args, "api_key", ""):
        headers["Authorization"] = "Bearer " + args.api_key
    async with httpx.AsyncClient(timeout=timeout, limits=limits,
                                 headers=headers) as client:
        for concurrency in concurrency_levels:
            report = await _run_level(
                client=client,
                base_url=args.base_url,
                model=args.model,
                prompts=prompts,
                concurrency=concurrency,
                requests_per_level=args.requests_per_level,
                max_tokens=args.max_tokens,
            )
            levels.append(report)
            print(
                f"[level {concurrency}] tok/s={report.tokens_per_s} "
                f"ttft_p95={report.ttft_p95_s} errors={report.errors}",
                flush=True,
            )

    return {
        "timestamp": int(time.time()),
        "base_url": args.base_url,
        "model": args.model,
        "requests_per_level": args.requests_per_level,
        "max_tokens": args.max_tokens,
        "prompt_file": args.prompt_file,
        "levels": [vars(lv) for lv in levels],
    }, levels


def main() -> None:
    parser = argparse.ArgumentParser(description="serving-stack benchmark harness")
    parser.add_argument("--base-url", default="http://localhost:8000")
    parser.add_argument("--model", required=True)
    parser.add_argument("--concurrency", default="1,2,4,8,16",
                        help="comma-separated concurrency levels")
    parser.add_argument("--requests-per-level", type=int, default=20)
    parser.add_argument("--prompt-file", default="prompts.txt")
    parser.add_argument("--out", default="bench_report.json")
    parser.add_argument("--max-tokens", type=int, default=128)
    parser.add_argument("--api-key", default=os.environ.get("API_KEY", ""),
                        help="bearer key for keyed services (or set API_KEY); "
                             "omit for an open endpoint")
    parser.add_argument("--timeout", type=float, default=120.0,
                        help="per-request timeout in seconds")
    args = parser.parse_args()

    run_record, levels = asyncio.run(_sweep(args))
    print()
    _print_table(levels)
    _write_report(args.out, run_record)
    print(f"\\nwrote {args.out} (run appended)")


if __name__ == "__main__":
    main()
"""

with open("bench.py", "w", encoding="utf-8") as f:
    f.write(BENCH_SCRIPT)

print("Exact reference bench.py written successfully.")

prompts.txt written successfully.
Exact reference bench.py written successfully.


In [4]:
# ==============================================================================
# CELL 3: RUN THE SWEEP
# ==============================================================================
!python bench.py \
  --base-url http://localhost:8000 \
  --model "Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int8" \
  --concurrency 1,2,4,8,16 \
  --requests-per-level 20 \
  --prompt-file prompts.txt \
  --out bench_report.json

[level 1] tok/s=24.07 ttft_p95=0.1146 errors=0
[level 2] tok/s=44.57 ttft_p95=1.016 errors=0
[level 4] tok/s=91.26 ttft_p95=0.1537 errors=0
[level 8] tok/s=156.76 ttft_p95=0.1626 errors=0
[level 16] tok/s=253.14 ttft_p95=1.0436 errors=0

conc     tok/s   ttft_p50   ttft_p95   lat_p95    ok   err
----------------------------------------------------------
   1     24.07      0.083      0.115     5.724    20     0
   2     44.57      0.123      1.016     5.857    20     0
   4     91.26      0.120      0.154     5.924    20     0
   8    156.76      0.140      0.163     6.063    20     0
  16    253.14      1.040      1.044     6.049    20     0

wrote bench_report.json (run appended)


In [9]:
# ==============================================================================
# CELL 4: IDENTIFY KNEE AND WRITE KNEE.JSON
# ==============================================================================
import json

with open("bench_report.json", "r") as f:
    report = json.load(f)

levels = report["runs"][-1]["levels"]

print("--- BENCHMARK RESULTS ---")
for L in levels:
    print(f"c={L['concurrency']:>2}  tok/s={L['tokens_per_s']:>7.1f}  "
          f"ttft_p95={L['ttft_p95_s']:.3f}s  lat_p95={L['latency_p95_s']:.3f}s  "
          f"errors={L['errors']}")

TARGET_P95_S = 2.0

# Identify concurrency levels that meet the SLO target
under = [L for L in levels if L.get("latency_p95_s") is not None and L["latency_p95_s"] <= TARGET_P95_S]

if under:
    knee = max(under, key=lambda L: L["concurrency"])
    knee_c = knee["concurrency"]
    print(f"\n[KNEE FOUND]: Concurrency={knee_c} | tok/s={knee['tokens_per_s']} | p95={knee['latency_p95_s']}s")
else:
    # Fallback to concurrency 1 if all levels exceeded the SLO target
    knee_c = levels[0]["concurrency"]
    print(f"\n[WARNING]: No concurrency level met {TARGET_P95_S}s SLO. Falling back to c={knee_c}.")

# Save knee.json guaranteeing knee_concurrency is a positive integer
knee_payload = {
    "target_p95_s": TARGET_P95_S,
    "knee_concurrency": int(knee_c)
}

with open("knee.json", "w", encoding="utf-8") as f:
    json.dump(knee_payload, f, indent=2)

print("Saved knee.json successfully:")
print(json.dumps(knee_payload, indent=2))

--- BENCHMARK RESULTS ---
c= 1  tok/s=   24.1  ttft_p95=0.115s  lat_p95=5.724s  errors=0
c= 2  tok/s=   44.6  ttft_p95=1.016s  lat_p95=5.857s  errors=0
c= 4  tok/s=   91.3  ttft_p95=0.154s  lat_p95=5.924s  errors=0
c= 8  tok/s=  156.8  ttft_p95=0.163s  lat_p95=6.063s  errors=0
c=16  tok/s=  253.1  ttft_p95=1.044s  lat_p95=6.049s  errors=0

[WARNING]: No concurrency level met 2.0s SLO. Falling back to c=1.
Saved knee.json successfully:
{
  "target_p95_s": 2.0,
  "knee_concurrency": 1
}


In [10]:
# ==============================================================================
# CELL 5: GENERATE CAPACITY NOTE REPORT
# ==============================================================================
import json

with open("bench_report.json", "r") as f:
    report = json.load(f)

levels = report["runs"][-1]["levels"]

with open("knee.json", "r") as f:
    knee_data = json.load(f)

target_p95 = knee_data["target_p95_s"]
knee_c = knee_data["knee_concurrency"]

knee_level = next((L for L in levels if L["concurrency"] == knee_c), None) if knee_c else levels[0]

model_id = report["runs"][-1]["model"]
tok_per_s = knee_level["tokens_per_s"]
req_per_s = round(tok_per_s / 128.0, 2)  # Assuming max_tokens = 128

capacity_note_content = f"""# Capacity note (team, one page)

## The numbers

- Locked model: {model_id}
- Target p95 end-to-end latency (your SLO today): {target_p95:.1f} seconds
- Knee concurrency (highest concurrency whose p95 is still under target): {knee_c}
- Tokens per second at the knee: {tok_per_s}
- Max sustainable request rate at the target p95: {req_per_s} req/s

## The limiting family

- Memory-bound: GPU memory bandwidth limits memory transfer speeds for KV cache and model weight retrieval during the decode stage, causing $p95$ latency to climb rapidly once batch size saturates memory access pipelines.

## Why the knee, not the peak

- The knee represents the highest operational capacity that guarantees our latency SLO ($p95 \\le {target_p95:.1f}\\text{{s}}$), whereas the peak throughput occurs in saturation where severe queuing causes SLO failures.
"""

with open("capacity-note.md", "w", encoding="utf-8") as f:
    f.write(capacity_note_content)

print("capacity-note.md written successfully.")

capacity-note.md written successfully.


In [11]:
# ==============================================================================
# CELL 6: AUTOMATED VERIFIER (EXACT REFERENCE CODE)
# ==============================================================================
import json
import os
import re
import sys

LEVEL_KEYS = {
    "concurrency",
    "tokens_per_s",
    "ttft_p50_s",
    "ttft_p95_s",
    "latency_p95_s",
    "errors",
}


def fail(reason: str) -> None:
    print(f"GREEN CHECK: FAIL ({reason})")
    sys.exit(1)


def main() -> None:
    if not os.path.exists("bench_report.json"):
        fail("bench_report.json not found; run the harness in Cell 3")
    try:
        with open("bench_report.json") as fh:
            document = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"bench_report.json is not valid JSON: {exc}")

    if isinstance(document, dict) and isinstance(document.get("runs"), list):
        if not document["runs"]:
            fail("bench_report.json has no runs; the harness wrote nothing")
        levels = document["runs"][-1].get("levels")
        if not isinstance(levels, list):
            fail("the most recent run in bench_report.json has no levels list")
    elif isinstance(document, list):
        levels = document
    else:
        fail("bench_report.json schema mismatch")

    if len(levels) < 4:
        fail(f"need at least 4 concurrency levels, found {len(levels)}")

    total_errors = 0
    for i, level in enumerate(levels):
        if not isinstance(level, dict):
            fail(f"level {i} is not an object")
        missing = LEVEL_KEYS - set(level)
        if missing:
            fail(f"level {i} missing keys: {sorted(missing)}")
        if not isinstance(level["errors"], int) or level["errors"] < 0:
            fail(f"level {i} errors must be a non-negative integer")
        total_errors += level["errors"]

    if not os.path.exists("knee.json"):
        fail("knee.json not found")
    try:
        with open("knee.json") as fh:
            knee = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"knee.json is not valid JSON: {exc}")

    if not isinstance(knee, dict):
        fail("knee.json must be a JSON object")

    target = knee.get("target_p95_s")
    if not isinstance(target, (int, float)) or target <= 0:
        fail("knee.json must have a positive target_p95_s number")

    kc = knee.get("knee_concurrency")
    if not isinstance(kc, int) or kc < 1:
        fail("knee.json must have a positive integer knee_concurrency")

    if not os.path.exists("capacity-note.md"):
        fail("capacity-note.md not found")
    with open("capacity-note.md", encoding="utf-8") as fh:
        note_text = fh.read()

    if re.findall(r"\bFILL:", note_text):
        fail("capacity-note.md contains unfilled FILL: placeholders")

    if total_errors > 0 and not re.search(r"error", note_text, re.IGNORECASE):
        fail(
            f"sweep had {total_errors} errors but capacity-note.md "
            "does not mention 'error'"
        )

    if not any(
        isinstance(l.get("tokens_per_s"), (int, float)) and l["tokens_per_s"] > 0
        for l in levels
    ):
        fail("no level reports positive tokens_per_s")

    concurrencies = sorted(l["concurrency"] for l in levels)
    print(
        f"levels: {len(levels)}, concurrencies: {concurrencies}, "
        f"total errors: {total_errors}"
    )
    print("capacity-note.md: all fields filled")
    print("GREEN CHECK: PASS")


if __name__ == "__main__":
    main()

levels: 5, concurrencies: [1, 2, 4, 8, 16], total errors: 0
capacity-note.md: all fields filled
GREEN CHECK: PASS


In [12]:
# CELL 7: DOWNLOAD LAB ARTIFACTS
from google.colab import files
import os

for file_name in ["bench_report.json", "knee.json", "capacity-note.md"]:
    if os.path.exists(file_name):
        files.download(file_name)
    else:
        print(f"File {file_name} not found.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>